# **Langkah 2: Melatih Model Baseline (8 Kategori) Binary**

###**2.1 Definisi Fungsi Pembantu - Memory Management (Helper Functions)**

In [ ]:
def get_memory_usage():
    """Mendapatkan penggunaan RAM sistem"""
    memory = psutil.virtual_memory()
    used_gb = (memory.total - memory.available) / (1024**3)
    total_gb = memory.total / (1024**3)
    available_gb = memory.available / (1024**3)
    percent = memory.percent
    return used_gb, total_gb, available_gb, percent

def print_memory_status(stage=""):
    """Print status memori"""
    used, total, available, percent = get_memory_usage()
    print(f"\n[Memory Status - {stage}]")
    print(f"  Used: {used:.2f} GB / {total:.2f} GB ({percent:.1f}%)")
    print(f"  Available: {available:.2f} GB")

def clear_memory():
    """Bersihkan memory"""
    gc.collect()
    print("  Memory cleared")

print(" Memory management functions defined!")

 Memory management functions defined!


###**2.2 Definisi Fungsi Evaluasi Model Binary Classification**

In [ ]:
def evaluate_binary_model(y_true, y_pred, y_pred_proba, category_name, model_name, training_time):
    """
    Evaluasi untuk SATU binary classifier (One category vs Rest)

    Parameters:
    -----------
    y_true : array-like (shape: n_samples)
        True binary labels (0 or 1)
    y_pred : array-like (shape: n_samples)
        Predicted binary labels (0 or 1)
    y_pred_proba : array-like (shape: n_samples, 2) or (shape: n_samples)
        Prediction probabilities
    category_name : str
        Nama kategori yang sedang dievaluasi
    model_name : str
        Nama model (RF, XGBoost, etc.)
    training_time : float
        Waktu training dalam detik

    Returns:
    --------
    results : dict
        Metrik evaluasi binary classification
    """
    # Handle probability format
    if len(y_pred_proba.shape) == 2:
        # Jika shape (n_samples, 2), ambil kolom ke-1 (probability untuk class 1)
        y_prob = y_pred_proba[:, 1]
    else:
        # Jika shape (n_samples,), langsung gunakan
        y_prob = y_pred_proba

    # Calculate metrics
    results = {
        'model_name': model_name,
        'category': category_name,
        'training_time_seconds': training_time,

        # Basic metrics
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),

        # Confusion matrix elements
        'tn': None, 'fp': None, 'fn': None, 'tp': None
    }

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        results['tn'] = int(tn)
        results['fp'] = int(fp)
        results['fn'] = int(fn)
        results['tp'] = int(tp)

    # Classification report
    report = classification_report(
        y_true, y_pred,
        target_names=['Negative', 'Positive'],
        output_dict=True,
        zero_division=0
    )

    return results, cm, report

print("evaluate_binary_model() function defined!")

evaluate_binary_model() function defined!


###**2.3 Definisi Fungsi Print Results**

In [ ]:
def print_binary_results(results):
    """Print hasil evaluasi binary classifier"""
    print(f"\n{'='*70}")
    print(f"BINARY CLASSIFIER: {results['model_name']} - {results['category']}")
    print(f"{'='*70}")
    print(f"Training Time: {results['training_time_seconds']:.2f} detik")
    print(f"\nMetrik:")
    print(f"  Accuracy  : {results['accuracy']:.4f}")
    print(f"  Precision : {results['precision']:.4f}")
    print(f"  Recall    : {results['recall']:.4f}")
    print(f"  F1-Score  : {results['f1_score']:.4f}")
    print(f"  MCC       : {results['mcc']:.4f}")

    if results['tp'] is not None:
        print(f"\nConfusion Matrix:")
        print(f"  TN={results['tn']:,}  FP={results['fp']:,}")
        print(f"  FN={results['fn']:,}  TP={results['tp']:,}")
    print(f"{'='*70}")


print("print_binary_results() function defined!")

print_binary_results() function defined!


###**2.4 Definisi Fungsi Print Metrik Agregat**

In [ ]:
def aggregate_binary_results(all_category_results):
    """
    Aggregate hasil dari semua binary classifiers

    Parameters:
    -----------
    all_category_results : list of dict
        List hasil evaluasi untuk setiap kategori

    Returns:
    --------
    aggregated : dict
        Metrik agregat (macro average)
    """
    metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'mcc']
    aggregated = {
        'avg_accuracy': np.mean([r['accuracy'] for r in all_category_results]),
        'avg_precision': np.mean([r['precision'] for r in all_category_results]),
        'avg_recall': np.mean([r['recall'] for r in all_category_results]),
        'avg_f1_score': np.mean([r['f1_score'] for r in all_category_results]),
        'avg_mcc': np.mean([r['mcc'] for r in all_category_results]),
        'total_training_time': sum([r['training_time_seconds'] for r in all_category_results]),
        'num_categories': len(all_category_results)
    }

    return aggregated

print("aggregate_binary_results() function defined!")

aggregate_binary_results() function defined!


In [ ]:
# Inisialisasi List untuk Menyimpan Hasil
all_results = []
print("\n All helper functions ready!")
print("Results storage initialized!")


 All helper functions ready!
Results storage initialized!


###**2.5 Load Kembali Hasil Pre-Processing Data**

In [ ]:
print_memory_status("Sebelum Load Data")

print("\nMemuat data terproses dari Google Drive...")

# Load fitur
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))

# Load nama kategori
with open(os.path.join(PROCESSED_DIR, 'category_names.json'), 'r') as f:
    category_names = json.load(f)

# Load binary labels dictionary
y_train_binary_dict = {}
y_test_binary_dict = {}

for category in category_names:
    safe_name = category.replace('/', '_')
    y_train_binary_dict[category] = np.load(
        os.path.join(PROCESSED_DIR, f'y_train_binary_{safe_name}.npy')
    )
    y_test_binary_dict[category] = np.load(
        os.path.join(PROCESSED_DIR, f'y_test_binary_{safe_name}.npy')
    )

print(f"\nData Binary OvR berhasil dimuat!")
print(f"  - X_train shape: {X_train.shape}")
print(f"  - X_test shape: {X_test.shape}")
print(f"  - Jumlah kategori (binary models): {len(category_names)}")
print(f"\nKategori yang diload: {category_names}")

print_memory_status("Setelah Load Data")


[Memory Status - Sebelum Load Data]
  Used: 2.77 GB / 50.99 GB (5.4%)
  Available: 48.22 GB

Memuat data terproses dari Google Drive...

Data Binary OvR berhasil dimuat!
  - X_train shape: (386274, 71)
  - X_test shape: (45319, 71)
  - Jumlah kategori (binary models): 8

Kategori yang diload: ['Benign', 'Brute_Force', 'DDoS', 'DoS', 'MITM/Spoofing', 'Malware/Mirai', 'Recon', 'Web_Based']

[Memory Status - Setelah Load Data]
  Used: 3.02 GB / 50.99 GB (5.9%)
  Available: 47.97 GB


###**2.6 Training Random Forest (Binary One-vs-Rest)**

In [ ]:
print_memory_status("Before RF OvR")

# Storage untuk semua hasil RF
rf_results_all_categories = []
rf_models = {}

# Loop untuk setiap kategori
for idx, category in enumerate(category_names, 1):
    print(f"\n{'='*70}")
    print(f"[{idx}/{len(category_names)}] Training RF untuk: {category}")
    print(f"{'='*70}")

    # Get binary labels
    y_train_cat = y_train_binary_dict[category]
    y_test_cat = y_test_binary_dict[category]

    # Check imbalance
    pos_ratio = y_train_cat.sum() / len(y_train_cat)
    print(f"Training set: {y_train_cat.sum():,} positive ({pos_ratio*100:.2f}%), "
          f"{len(y_train_cat) - y_train_cat.sum():,} negative")

    # Initialize RF with class_weight for imbalance
    start_time = time.time()

    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
        verbose=0  # Matikan verbose untuk training multiple models
    )

    print(f" Training Random Forest...")
    rf_model.fit(X_train_scaled, y_train_cat)
    training_time = time.time() - start_time

    print(f" Training selesai: {training_time:.2f} detik")

    # Prediction
    print(" Predicting...")
    y_pred_cat = rf_model.predict(X_test_scaled)
    y_pred_proba_cat = rf_model.predict_proba(X_test_scaled)

    # Evaluation
    results, cm, report = evaluate_binary_model(
        y_test_cat, y_pred_cat, y_pred_proba_cat,
        category, "Random Forest", training_time
    )

    print_binary_results(results)

    # Save results
    rf_results_all_categories.append(results)
    rf_models[category] = rf_model

    # Save individual model
    safe_name = category.replace('/', '_')
    with open(os.path.join(MODELS_DIR, f'rf_binary_{safe_name}.pkl'), 'wb') as f:
        pickle.dump(rf_model, f)

    print(f" Model saved: rf_binary_{safe_name}.pkl")

    clear_memory()

# Aggregate results
print(f"\n{'='*70}")
print("RANDOM FOREST - AGGREGATED RESULTS (ALL 8 CATEGORIES)")
print(f"{'='*70}")

rf_aggregated = aggregate_binary_results(rf_results_all_categories)
print(f"\nAverage Metrics (Macro Average across 8 categories):")
print(f"  Accuracy  : {rf_aggregated['avg_accuracy']:.4f}")
print(f"  Precision : {rf_aggregated['avg_precision']:.4f}")
print(f"  Recall    : {rf_aggregated['avg_recall']:.4f}")
print(f"  F1-Score  : {rf_aggregated['avg_f1_score']:.4f}")
print(f"  MCC       : {rf_aggregated['avg_mcc']:.4f}")
print(f"\nTotal Training Time: {rf_aggregated['total_training_time']:.2f} seconds "
      f"({rf_aggregated['total_training_time']/60:.2f} minutes)")

# Save aggregated results
rf_aggregated['model_name'] = 'Random Forest (Binary OvR)'
rf_aggregated['category_results'] = rf_results_all_categories
all_results.append(rf_aggregated)

with open(os.path.join(RESULTS_DIR, 'rf_binary_ovr_results.json'), 'w') as f:
    json.dump({
        'aggregated': rf_aggregated,
        'per_category': rf_results_all_categories
    }, f, indent=4, default=int)

print(f"\n All RF OvR results saved!")
print_memory_status("After RF OvR")


[Memory Status - Before RF OvR]
  Used: 5.30 GB / 50.99 GB (10.4%)
  Available: 45.69 GB

[1/8] Training RF untuk: Benign
Training set: 163,578 positive (42.35%), 222,696 negative
 Training Random Forest...
 Training selesai: 14.19 detik
 Predicting...

BINARY CLASSIFIER: Random Forest - Benign
Training Time: 14.19 detik

Metrik:
  Accuracy  : 0.9954
  Precision : 0.9874
  Recall    : 0.9995
  F1-Score  : 0.9934
  MCC       : 0.9899

Confusion Matrix:
  TN=29,384  FP=201
  FN=8  TP=15,726
 Model saved: rf_binary_Benign.pkl
  Memory cleared

[2/8] Training RF untuk: Brute_Force
Training set: 4,055 positive (1.05%), 382,219 negative
 Training Random Forest...
 Training selesai: 13.11 detik
 Predicting...

BINARY CLASSIFIER: Random Forest - Brute_Force
Training Time: 13.11 detik

Metrik:
  Accuracy  : 0.9999
  Precision : 1.0000
  Recall    : 0.9930
  F1-Score  : 0.9965
  MCC       : 0.9964

Confusion Matrix:
  TN=44,748  FP=0
  FN=4  TP=567
 Model saved: rf_binary_Brute_Force.pkl
  Memo

###**2.7 Training XGBoost (Binary One-vs-Rest)**

In [ ]:
print_memory_status("Before XGBoost OvR")

xgb_results_all_categories = []
xgb_models = {}

for idx, category in enumerate(category_names, 1):
    print(f"\n{'='*70}")
    print(f"[{idx}/{len(category_names)}] Training XGBoost untuk: {category}")
    print(f"{'='*70}")

    y_train_cat = y_train_binary_dict[category]
    y_test_cat = y_test_binary_dict[category]

    start_time = time.time()

    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',  # BINARY objective!
        eval_metric='logloss',
        tree_method='hist',
        device='cuda',
        random_state=42,
        n_jobs=-1
    )

    print(f" Training XGBoost...")
    try:
        xgb_model.fit(X_train_scaled, y_train_cat, verbose=False)
    except Exception as e:
        print(f" GPU failed: {e}, switching to CPU...")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(X_train_scaled, y_train_cat, verbose=False)

    training_time = time.time() - start_time
    print(f" Training selesai: {training_time:.2f} detik")

    # Prediction
    y_pred_cat = xgb_model.predict(X_test_scaled)
    y_pred_proba_cat = xgb_model.predict_proba(X_test_scaled)

    # Evaluation
    results, cm, report = evaluate_binary_model(
        y_test_cat, y_pred_cat, y_pred_proba_cat,
        category, "XGBoost", training_time
    )

    print_binary_results(results)

    xgb_results_all_categories.append(results)
    xgb_models[category] = xgb_model

    # Save
    safe_name = category.replace('/', '_')
    xgb_model.save_model(os.path.join(MODELS_DIR, f'xgb_binary_{safe_name}.json'))
    print(f" Model saved: xgb_binary_{safe_name}.json")

    clear_memory()

# Aggregate
print(f"\n{'='*70}")
print("XGBOOST - AGGREGATED RESULTS")
print(f"{'='*70}")

xgb_aggregated = aggregate_binary_results(xgb_results_all_categories)
print(f"\nAverage Metrics:")
print(f"  Accuracy  : {xgb_aggregated['avg_accuracy']:.4f}")
print(f"  Precision : {xgb_aggregated['avg_precision']:.4f}")
print(f"  Recall    : {xgb_aggregated['avg_recall']:.4f}")
print(f"  F1-Score  : {xgb_aggregated['avg_f1_score']:.4f}")
print(f"  MCC       : {xgb_aggregated['avg_mcc']:.4f}")
print(f"\nTotal Training Time: {xgb_aggregated['total_training_time']:.2f}s")

xgb_aggregated['model_name'] = 'XGBoost (Binary OvR)'
xgb_aggregated['category_results'] = xgb_results_all_categories
all_results.append(xgb_aggregated)

with open(os.path.join(RESULTS_DIR, 'xgb_binary_ovr_results.json'), 'w') as f:
    json.dump({
        'aggregated': xgb_aggregated,
        'per_category': xgb_results_all_categories
    }, f, indent=4, default=int)

print_memory_status("After XGBoost OvR")


[Memory Status - Before XGBoost OvR]
  Used: 5.48 GB / 50.99 GB (10.7%)
  Available: 45.51 GB

[1/8] Training XGBoost untuk: Benign
 Training XGBoost...
 Training selesai: 2.30 detik

BINARY CLASSIFIER: XGBoost - Benign
Training Time: 2.30 detik

Metrik:
  Accuracy  : 0.9966
  Precision : 0.9918
  Recall    : 0.9984
  F1-Score  : 0.9951
  MCC       : 0.9925

Confusion Matrix:
  TN=29,455  FP=130
  FN=25  TP=15,709
 Model saved: xgb_binary_Benign.json
  Memory cleared

[2/8] Training XGBoost untuk: Brute_Force
 Training XGBoost...
 Training selesai: 1.45 detik

BINARY CLASSIFIER: XGBoost - Brute_Force
Training Time: 1.45 detik

Metrik:
  Accuracy  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000
  F1-Score  : 1.0000
  MCC       : 1.0000

Confusion Matrix:
  TN=44,748  FP=0
  FN=0  TP=571
 Model saved: xgb_binary_Brute_Force.json
  Memory cleared

[3/8] Training XGBoost untuk: DDoS
 Training XGBoost...
 Training selesai: 1.72 detik

BINARY CLASSIFIER: XGBoost - DDoS
Training Time: 1.7

###**2.8 Training LightGBM (Binary One-vs-Rest)**

In [ ]:
lgb_results_all_categories = []
lgb_models = {}

for idx, category in enumerate(category_names, 1):
    print(f"\n{'='*70}")
    print(f"[{idx}/{len(category_names)}] Training LightGBM untuk: {category}")
    print(f"{'='*70}")

    y_train_cat = y_train_binary_dict[category]
    y_test_cat = y_test_binary_dict[category]

    start_time = time.time()

    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary',  # BINARY objective!
        device='gpu',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    print(f" Training LightGBM...")
    try:
        lgb_model.fit(X_train_scaled, y_train_cat)
    except Exception as e:
        print(f" GPU failed: {e}, switching to CPU...")
        lgb_model.set_params(device='cpu')
        lgb_model.fit(X_train_scaled, y_train_cat)

    training_time = time.time() - start_time
    print(f" Training selesai: {training_time:.2f} detik")

    # Prediction
    y_pred_cat = lgb_model.predict(X_test_scaled)
    y_pred_proba_cat = lgb_model.predict_proba(X_test_scaled)

    # Evaluation
    results, cm, report = evaluate_binary_model(
        y_test_cat, y_pred_cat, y_pred_proba_cat,
        category, "LightGBM", training_time
    )

    print_binary_results(results)

    lgb_results_all_categories.append(results)
    lgb_models[category] = lgb_model

    # Save
    safe_name = category.replace('/', '_')
    with open(os.path.join(MODELS_DIR, f'lgb_binary_{safe_name}.pkl'), 'wb') as f:
        pickle.dump(lgb_model, f)
    print(f" Model saved: lgb_binary_{safe_name}.pkl")

    clear_memory()

# Aggregate
print(f"\n{'='*70}")
print("LIGHTGBM - AGGREGATED RESULTS")
print(f"{'='*70}")

lgb_aggregated = aggregate_binary_results(lgb_results_all_categories)
print(f"\nAverage Metrics:")
print(f"  Accuracy  : {lgb_aggregated['avg_accuracy']:.4f}")
print(f"  Precision : {lgb_aggregated['avg_precision']:.4f}")
print(f"  Recall    : {lgb_aggregated['avg_recall']:.4f}")
print(f"  F1-Score  : {lgb_aggregated['avg_f1_score']:.4f}")
print(f"  MCC       : {lgb_aggregated['avg_mcc']:.4f}")
print(f"\nTotal Training Time: {lgb_aggregated['total_training_time']:.2f}s")

lgb_aggregated['model_name'] = 'LightGBM (Binary OvR)'
lgb_aggregated['category_results'] = lgb_results_all_categories
all_results.append(lgb_aggregated)

with open(os.path.join(RESULTS_DIR, 'lgb_binary_ovr_results.json'), 'w') as f:
    json.dump({
        'aggregated': lgb_aggregated,
        'per_category': lgb_results_all_categories
    }, f, indent=4, default=int)

print_memory_status("After LightGBM OvR")


[1/8] Training LightGBM untuk: Benign
 Training LightGBM...
 GPU failed: Check failed: (best_split_info.right_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 862 .
, switching to CPU...
 Training selesai: 9.23 detik

BINARY CLASSIFIER: LightGBM - Benign
Training Time: 9.23 detik

Metrik:
  Accuracy  : 0.9966
  Precision : 0.9911
  Recall    : 0.9993
  F1-Score  : 0.9952
  MCC       : 0.9926

Confusion Matrix:
  TN=29,444  FP=141
  FN=11  TP=15,723
 Model saved: lgb_binary_Benign.pkl
  Memory cleared

[2/8] Training LightGBM untuk: Brute_Force
 Training LightGBM...
 Training selesai: 2.33 detik

BINARY CLASSIFIER: LightGBM - Brute_Force
Training Time: 2.33 detik

Metrik:
  Accuracy  : 0.9996
  Precision : 0.9964
  Recall    : 0.9755
  F1-Score  : 0.9858
  MCC       : 0.9857

Confusion Matrix:
  TN=44,746  FP=2
  FN=14  TP=557
 Model saved: lgb_binary_Brute_Force.pkl
  Memory cleared

[3/8] Training LightGBM untuk: DDoS
 Training LightGBM...
 Train

###**2.9 Training Deep Neural Network (DNN)**

In [ ]:
print_memory_status("Sebelum Training DNN")

# Clear memory sebelum training
clear_memory()

# Get number of features
num_features = X_train.shape[1]
# The num_classes line is not needed for binary OvR DNN, remove to prevent NameError
# num_classes = len(np.unique(y_train))

print(f"\nParameter DNN:")
print(f"  - Input features: {num_features}")
print(f"  - Output classes: 2 (for binary classification)")
print(f"  - Train samples: {X_train.shape[0]:,}")

###2.9 Training DNN (Binary One-vs-Rest)
def create_binary_dnn_model(num_features):
    """
    DNN untuk BINARY classification (1 output neuron dengan sigmoid)
    """
    model = keras.Sequential([
        layers.Input(shape=(num_features,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),

        layers.Dense(16, activation='relu'),

        layers.Dense(1, activation='sigmoid')  # BINARY output!
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',  # BINARY loss!
        metrics=['accuracy']
    )

    return model

# Training loop (sama seperti RF/XGB)
dnn_results_all_categories = []
dnn_models = {}

for idx, category in enumerate(category_names, 1):
    print(f"\n[{idx}/{len(category_names)}] Training DNN untuk: {category}")

    y_train_cat = y_train_binary_dict[category]
    y_test_cat = y_test_binary_dict[category]

    # Calculate class weight
    neg_count = len(y_train_cat) - y_train_cat.sum()
    pos_count = y_train_cat.sum()
    class_weight = {0: 1.0, 1: neg_count/pos_count if pos_count > 0 else 1.0}

    dnn_model = create_binary_dnn_model(num_features)

    checkpoint = ModelCheckpoint(
        os.path.join(CHECKPOINT_DIR, f'dnn_binary_{category.replace("/", "_")}.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )

    start_time = time.time()

    history = dnn_model.fit(
        X_train, y_train_cat,
        validation_split=0.1,
        epochs=50,
        batch_size=256,
        callbacks=[checkpoint, EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=0
    )

    training_time = time.time() - start_time

    # Prediction (binary)
    y_pred_proba_cat = dnn_model.predict(X_test, verbose=0).flatten()
    y_pred_cat = (y_pred_proba_cat > 0.5).astype(int)

    # Evaluation
    results, cm, report = evaluate_binary_model(
        y_test_cat, y_pred_cat, y_pred_proba_cat,
        category, "DNN", training_time
    )

    print_binary_results(results)

    dnn_results_all_categories.append(results)
    dnn_models[category] = dnn_model

    # Save individual DNN model
    safe_name = category.replace('/', '_')
    model_save_path = os.path.join(MODELS_DIR, f'dnn_binary_{safe_name}.keras')
    dnn_model.save(model_save_path)
    print(f" Model saved: dnn_binary_{safe_name}.keras")

    clear_memory()

# Aggregate DNN results
dnn_aggregated = aggregate_binary_results(dnn_results_all_categories)
print(f"\nDNN Average Metrics:")
print(f"  Accuracy: {dnn_aggregated['avg_accuracy']:.4f}")
print(f"  Precision : {dnn_aggregated['avg_precision']:.4f}")
print(f"  Recall    : {dnn_aggregated['avg_recall']:.4f}")
print(f"  F1-Score: {dnn_aggregated['avg_f1_score']:.4f}")
print(f"  MCC       : {dnn_aggregated['avg_mcc']:.4f}")
print(f"\nTotal Training Time: {dnn_aggregated['total_training_time']:.2f}s")

dnn_aggregated['model_name'] = 'DNN (Binary OvR)'
dnn_aggregated['category_results'] = dnn_results_all_categories
all_results.append(dnn_aggregated)

with open(os.path.join(RESULTS_DIR, 'dnn_binary_ovr_results.json'), 'w') as f:
    json.dump({
        'aggregated': dnn_aggregated,
        'per_category': dnn_results_all_categories
    }, f, indent=4, default=int)

print_memory_status("After DNN OvR")


[Memory Status - Sebelum Training DNN]
  Used: 3.13 GB / 50.99 GB (6.1%)
  Available: 47.86 GB
  Memory cleared

Parameter DNN:
  - Input features: 71
  - Output classes: 2 (for binary classification)
  - Train samples: 386,274

[1/8] Training DNN untuk: Benign

BINARY CLASSIFIER: DNN - Benign
Training Time: 123.43 detik

Metrik:
  Accuracy  : 0.9897
  Precision : 0.9824
  Recall    : 0.9881
  F1-Score  : 0.9853
  MCC       : 0.9774

Confusion Matrix:
  TN=29,307  FP=278
  FN=187  TP=15,547
 Model saved: dnn_binary_Benign.keras
  Memory cleared

[2/8] Training DNN untuk: Brute_Force

BINARY CLASSIFIER: DNN - Brute_Force
Training Time: 125.90 detik

Metrik:
  Accuracy  : 0.9979
  Precision : 0.9897
  Recall    : 0.8406
  F1-Score  : 0.9091
  MCC       : 0.9111

Confusion Matrix:
  TN=44,743  FP=5
  FN=91  TP=480
 Model saved: dnn_binary_Brute_Force.keras
  Memory cleared

[3/8] Training DNN untuk: DDoS

BINARY CLASSIFIER: DNN - DDoS
Training Time: 159.17 detik

Metrik:
  Accuracy  : 0.